<a href="https://colab.research.google.com/github/woodstock0513/-20262R0136COSE34102/blob/main/kuccthon_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 사투리 서비스 백엔드 실행
위에서 아래로 한 번씩 실행합니다. Colab 비밀정보에 `SUPABASE_URL`, `SUPABASE_KEY`, `NGROK_AUTHTOKEN`을 등록해 두세요. `dialect_fastapi_starter.zip` 파일을 업로드합니다.

In [ ]:
from google.colab import files
from pathlib import Path
import subprocess

uploaded = files.upload()
zip_name = next((n for n in uploaded if n.endswith(".zip")), None)
if not zip_name:
    raise ValueError("dialect_fastapi_starter.zip 파일을 업로드하세요.")
subprocess.run(["unzip", "-o", "-q", zip_name], check=True)
subprocess.run(["pip", "-q", "install", "-r", "dialect_api/requirements.txt", "requests"], check=True)
subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"], check=True)
print("백엔드 설치 완료")

Saving dialect_fastapi_starter.zip to dialect_fastapi_starter.zip
백엔드 설치 완료


## 설정
AI 서버 주소가 바뀌었다면 `AI_URL`을 수정하세요. 프론트엔드의 실제 주소를 알게 되면 `FRONTEND_ORIGIN`에 입력한 후 서버를 다시 실행하세요.

In [ ]:
import os
import secrets
from google.colab import userdata

AI_URL = "https://bcf9-203-251-135-23.ngrok-free.app"
FRONTEND_ORIGIN = ""

os.environ["SUPABASE_URL"] = userdata.get("SUPABASE_URL")
os.environ["SUPABASE_KEY"] = userdata.get("SUPABASE_KEY")
os.environ["NGROK_AUTHTOKEN"] = userdata.get("NGROK_AUTHTOKEN")
os.environ["AI_API_URL"] = AI_URL.rstrip("/")
os.environ["ADMIN_KEY"] = secrets.token_urlsafe(32)
os.environ["ALLOWED_ORIGINS"] = ",".join(filter(None, [
    "http://localhost:3000", "http://localhost:5173", FRONTEND_ORIGIN.rstrip("/")
]))
print("설정 완료")

설정 완료


## DNA 분석 API 적용
업로드 음성을 자동으로 16kHz WAV로 변환하고 AI 분석 결과를 Supabase에 저장합니다. 최고 점수 50% 미만이면 재녹음을 권장합니다.

In [ ]:
from pathlib import Path

app_path = Path("dialect_api/app.py")
code = app_path.read_text(encoding="utf-8")

api_code = r'''

@app.post("/dna/analyze")
def analyze_dialect(audio: UploadFile = File(...), session_id: str = Form(...)):
    import subprocess
    import tempfile
    import requests
    from pathlib import Path

    if not 8 <= len(session_id) <= 100:
        raise HTTPException(400, "session_id 형식 오류")

    raw = audio.file.read(3 * 1024 * 1024 + 1)
    if not raw or len(raw) > 3 * 1024 * 1024:
        raise HTTPException(413, "음성 파일은 3MB 이하만 가능합니다")

    with tempfile.TemporaryDirectory() as temp_dir:
        source = Path(temp_dir) / "input_audio"
        wav = Path(temp_dir) / "converted.wav"
        source.write_bytes(raw)

        try:
            subprocess.run([
                "ffmpeg", "-nostdin", "-y", "-loglevel", "error",
                "-i", str(source), "-t", "60", "-ar", "16000",
                "-ac", "1", "-c:a", "pcm_s16le", str(wav)
            ], check=True, capture_output=True, timeout=30)
        except (subprocess.SubprocessError, OSError):
            raise HTTPException(400, "음성 파일 변환 실패")

        ai_url = os.environ.get("AI_API_URL", "").rstrip("/")
        if not ai_url:
            raise HTTPException(503, "AI 서버 주소 미설정")

        try:
            with wav.open("rb") as f:
                response = requests.post(
                    ai_url + "/predict",
                    files={"audio": ("recording.wav", f, "audio/wav")},
                    headers={"ngrok-skip-browser-warning": "true"},
                    timeout=180
                )
            response.raise_for_status()
            results = response.json()["results"]
            if not results:
                raise ValueError("AI 응답에 결과 없음")
            best = max(results, key=lambda item: float(item["prob"]))
            scores = {item["name"]: float(item["prob"]) for item in results}
            confidence = float(best["prob"])
        except (requests.RequestException, ValueError, KeyError, TypeError):
            raise HTTPException(502, "AI 서버 응답 오류")

    saved = db.table("dna_analyses").insert({
        "session_id": session_id,
        "predicted_region": best["name"],
        "confidence": confidence,
        "scores": scores,
        "model_version": "wav2vec-bert-korean-dialect-recognition"
    }).execute().data[0]

    return {
        "analysis_id": saved["id"],
        "predicted_region": best["name"],
        "confidence": confidence,
        "scores": scores,
        "retry_recommended": confidence < 0.50,
        "donation_eligible": confidence >= 0.90
    }
'''

if '@app.post("/dna/analyze")' in code:
    code = code[:code.index('@app.post("/dna/analyze")')].rstrip()
app_path.write_text(code + "\n" + api_code, encoding="utf-8")
print("DNA 분석 API 적용 완료 (재녹음 기준 50%)")

DNA 분석 API 적용 완료 (재녹음 기준 50%)


## 서버 실행 및 외부 주소 발급
이 셀을 실행하면 API 주소가 출력됩니다. **프론트엔드 담당자에게는 이 백엔드 주소를 전달**하세요.

In [ ]:
import subprocess
import time
import requests
from pathlib import Path
from pyngrok import ngrok

subprocess.run(
    ["fuser", "-k", "8000/tcp"],
    check=False,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

server = subprocess.Popen(
    ["uvicorn", "dialect_api.app:app",
     "--host", "0.0.0.0", "--port", "8000"],
    stdout=open("/tmp/dialect_api.log", "w"),
    stderr=subprocess.STDOUT
)

time.sleep(3)

try:
    check = requests.get("http://127.0.0.1:8000/health", timeout=10)
    check.raise_for_status()
except Exception:
    print(Path("/tmp/dialect_api.log").read_text()[-2000:])
    raise

tunnels = ngrok.get_tunnels()
existing = next(
    (t for t in tunnels if str(t.config.get("addr", "")).endswith(":8000")),
    None
)

if existing:
    backend_url = existing.public_url
else:
    backend_url = ngrok.connect(8000, "http").public_url

print("백엔드:", backend_url)
print("DNA 분석 API:", backend_url + "/dna/analyze")
print("API 문서:", backend_url + "/docs")

백엔드: https://gleeful-backlog-justifier.ngrok-free.dev
DNA 분석 API: https://gleeful-backlog-justifier.ngrok-free.dev/dna/analyze
API 문서: https://gleeful-backlog-justifier.ngrok-free.dev/docs


## 선택: DNA 분석 테스트
필요할 때만 실행하세요. 분석 결과만 데이터베이스에 저장되며 음성 파일은 저장하지 않습니다.

In [ ]:
import uuid
import mimetypes
from google.colab import files

uploaded = files.upload()
filename = next(iter(uploaded))
response = requests.post(
    "http://127.0.0.1:8000/dna/analyze",
    data={"session_id": str(uuid.uuid4())},
    files={"audio": (
        filename,
        uploaded[filename],
        mimetypes.guess_type(filename)[0] or "application/octet-stream"
    )},
    timeout=240
)
print("상태 코드:", response.status_code)
print(response.json())

Saving test 음성.m4a to test 음성.m4a
상태 코드: 200
{'analysis_id': 'c943c9f3-2d49-4d3d-8b7d-94b76cb434fd', 'predicted_region': '전라도', 'confidence': 0.24766141176223755, 'scores': {'전라도': 0.24766141176223755, '서울·경기': 0.23609709739685059, '경상도': 0.22857995331287384, '제주도': 0.17513591051101685, '강원도': 0.06801658123731613, '충청도': 0.04450908675789833}, 'retry_recommended': True, 'donation_eligible': False}


In [ ]:
from pyngrok import ngrok

ngrok.kill()
print("현재 Colab의 ngrok 프로세스 종료 완료")

현재 Colab의 ngrok 프로세스 종료 완료


In [ ]:
from pyngrok import ngrok

print("현재 Colab의 터널:")
for tunnel in ngrok.get_tunnels():
    print(tunnel.public_url, tunnel.config)

현재 Colab의 터널:
https://gleeful-backlog-justifier.ngrok-free.dev {'addr': 'http://localhost:8000', 'inspect': True}


In [17]:
import requests

BACKEND_URL = "https://gleeful-backlog-justifier.ngrok-free.dev"
AI_URL = "https://bcf9-203-251-135-23.ngrok-free.app"

for name, url in [
    ("백엔드", BACKEND_URL + "/health"),
    ("AI 서버", AI_URL + "/openapi.json")
]:
    try:
        r = requests.get(
            url,
            headers={"ngrok-skip-browser-warning": "true"},
            timeout=15
        )
        print(name, "상태 코드:", r.status_code)
        print("응답:", r.text[:200])
    except Exception as e:
        print(name, "연결 실패:", str(e))

백엔드 상태 코드: 200
응답: {"ok":true}
AI 서버 상태 코드: 404
응답: <!DOCTYPE html>
<html class="h-full" lang="en-US" dir="ltr">
  <head>
    <meta charset="utf-8">
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <link rel="preload" href="


In [ ]:
import requests
import uuid

filename = next(iter(uploaded))

response = requests.post(
    "http://127.0.0.1:8000/dna/analyze",
    data={"session_id": str(uuid.uuid4())},
    files={"audio": (filename, uploaded[filename])},
    timeout=240
)

print("상태 코드:", response.status_code)
print("응답:", response.text[:1500])

상태 코드: 200
응답: {"analysis_id":"83f77e9f-410b-447d-96c6-4225144c9333","predicted_region":"전라도","confidence":0.24766141176223755,"scores":{"전라도":0.24766141176223755,"서울·경기":0.23609709739685059,"경상도":0.22857995331287384,"제주도":0.17513591051101685,"강원도":0.06801658123731613,"충청도":0.04450908675789833},"retry_recommended":true,"donation_eligible":false}
